# Step 11: Three classic CNN designs
LeNet style, VGG style and ResNet style, as 1-D networks over the 25 frames of each second, trained the same way and compared on the same random split (development stage). The winner (VGG style) is the CNN used in the final evaluation (step 15).
*Same results as `src/11_cnn_architectures.py`.*

**In short** (what this notebook shows)

- **What:** Three CNN designs, LeNet, VGG and ResNet style (1D convolutions over the 25 frames).
- **Needs the NoXi data?** Yes. Random split (development step).
- **Result / use:** VGG style works best (about 0.86). Shows convolution, pooling and residual connections.

### Dataset
**NoXi** (NOvice eXpert Interaction) - multimodal recordings of expert-novice conversations with continuous engagement labels.
- Dataset page (MultiMediate): https://multimediate.perceptualui.org/datasets/Dataset_NoXi/ (if it is offline, use the links below)
- Paper: A. Cafaro et al., "The NoXi Database: Multimodal Recordings of Mediated Novice-Expert Interactions", ACM ICMI 2017: https://eprints.nottingham.ac.uk/45827 (DOI https://doi.org/10.1145/3136755.3136780)
- Extension NoXi+J (2024): https://arxiv.org/abs/2409.13726
- Data access: University of Augsburg, Human-Centered AI lab - noxi@hcai.eu
- Used here: **24 conversations** (English 026, 027, 028, 029, 030, 058, 062, 063; French 001, 007, 009, 011, 013, 014, 020, 021; German 068, 070, 072, 073, 074, 075, 076, 077) from the NoXi train set:
  eGeMAPS voice features, OpenFace 2.0 face features, engagement labels (25 per second) and the pretrained-Transformer embeddings shipped with the corpus.
- Licence: NoXi is available for research under an End-User Licence Agreement (signed by the supervisor). The data is **not included** here and must not be shared;
  request access on the dataset page and place the downloaded zips as described in `01_extract_data`.

Go to the project folder.

In [2]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"     # hide TensorFlow messages
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"    # same maths settings as the original script
if os.getcwd().endswith("notebooks"):
    os.chdir("..")                           # work from the project folder

**Imports.**

In [3]:
import numpy as np
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
keras.utils.set_random_seed(42)

In [4]:
def ccc(y_true, y_pred):
    # Concordance Correlation Coefficient: 1 = perfect agreement
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    covariance = np.mean((y_true - y_true.mean()) * (y_pred - y_pred.mean()))
    return 2 * covariance / (y_true.var() + y_pred.var() + (y_true.mean() - y_pred.mean()) ** 2 + 1e-9)

**Load the sequences, split 60 / 20 / 20, scale every frame** (scaler fitted on training frames only).

In [5]:
y = pd.read_parquet("data/features/noxi_engagement.parquet", columns=["engagement"])["engagement"].to_numpy("float32")
X = np.load("data/features/noxi_sequences.npy", mmap_mode="r")   # read from disk piece by piece        # windows x 25 frames x 802
rest, test = train_test_split(np.arange(len(y)), test_size=0.2, random_state=42)
train, val = train_test_split(rest, test_size=0.25, random_state=42)
scaler = StandardScaler().fit(X[train].reshape(-1, 802))


def prepare(rows):
    return scaler.transform(X[rows].reshape(-1, 802)).reshape(len(rows), 25, 802).astype("float32")


X_train, X_val, X_test = prepare(train), prepare(val), prepare(test)
y_train, y_val, y_test = y[train], y[val], y[test]
del X

**LeNet style:** one convolution per block, wider kernel (5 frames).

In [6]:
def lenet():
    return keras.Sequential([
        layers.Input(shape=(25, 802)),
        layers.Conv1D(32, 5, activation="relu"),
        layers.MaxPooling1D(2),
        layers.Conv1D(64, 5, activation="relu"),
        layers.MaxPooling1D(2),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.1),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.1),
        layers.Dense(1),
    ], name="LeNet_style")

**VGG style:** two small (3-frame) convolutions stacked before each pooling.

In [7]:
def vgg():
    return keras.Sequential([
        layers.Input(shape=(25, 802)),
        layers.Conv1D(32, 3, padding="same", activation="relu"),
        layers.Conv1D(32, 3, padding="same", activation="relu"),
        layers.MaxPooling1D(2),
        layers.Conv1D(64, 3, padding="same", activation="relu"),
        layers.Conv1D(64, 3, padding="same", activation="relu"),
        layers.MaxPooling1D(2),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.1),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.1),
        layers.Dense(1),
    ], name="VGG_style")

**ResNet style:** residual blocks. The input skips over two convolutions and is added back (skip connection), so the block only learns the change.

In [8]:
def residual_block(x, filters):
    shortcut = x
    x = layers.Conv1D(filters, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv1D(filters, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([shortcut, x])          # the skip connection
    return layers.Activation("relu")(x)


def resnet():
    inp = layers.Input(shape=(25, 802))
    x = layers.Conv1D(64, 5, padding="same", activation="relu")(inp)
    x = layers.MaxPooling1D(2)(x)
    x = residual_block(x, 64)
    x = residual_block(x, 64)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.1)(x)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(1)(x)
    return keras.Model(inp, out, name="ResNet_style")

**Train all three the same way** (same seed, 20 epochs, Adam) and score them.

In [9]:
rows = []
for build in [lenet, vgg, resnet]:
    keras.utils.set_random_seed(42)          # same start for every design
    model = build()
    model.compile(optimizer=keras.optimizers.Adam(0.001), loss="mse")
    print(f"Training {model.name} ({model.count_params():,} parameters) ...")
    model.fit(X_train, y_train, epochs=20, batch_size=128, verbose=0)
    val_ccc = ccc(y_val, model.predict(X_val, verbose=0).flatten())
    pred = model.predict(X_test, verbose=0).flatten()
    rows.append({"model": model.name, "params": model.count_params(), "val_ccc": val_ccc,
                 "test_ccc": ccc(y_test, pred), "pearson": np.corrcoef(y_test, pred)[0, 1],
                 "mae": mean_absolute_error(y_test, pred)})
    print(f"  val CCC {val_ccc:.3f} | test CCC {rows[-1]['test_ccc']:.3f}")

Training LeNet_style (171,681 parameters) ...


  val CCC 0.619 | test CCC 0.819
Training VGG_style (156,289 parameters) ...


  val CCC 0.859 | test CCC 0.861
Training ResNet_style (311,361 parameters) ...


  val CCC 0.761 | test CCC 0.783


**Results** (best first) and comparison with the saved result (`reports/cnn_architectures.csv`).

In [10]:
results = pd.DataFrame(rows).sort_values("test_ccc", ascending=False).reset_index(drop=True)
print(results.round(3).to_string(index=False))
results.to_csv("reports/cnn_architectures.csv", index=False)
print("saved reports/cnn_architectures.csv")

       model  params  val_ccc  test_ccc  pearson   mae
   VGG_style  156289    0.859     0.861    0.866 0.054
 LeNet_style  171681    0.619     0.819    0.837 0.060
ResNet_style  311361    0.761     0.783    0.838 0.067
saved reports/cnn_architectures.csv
